In [1]:
import os
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
#if you need an API Key from OpenAI
#https://platform.openai.com/account/api-keys

from openai import OpenAI
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv(), override=True)

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [3]:
client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

prompt = """
    You're a tool for generating product category names based on the tags for each category.
    
    CRITICAL CONSTRAINT (read first): across ALL generated category names, no word may
    appear more than once (case-insensitive; ignore connectors like "&", "and", "of", "for").
    This is the hardest requirement and takes priority over how natural or generic a name sounds.

    Input: a JSON object where each key is a category ID (string) and each value \
    is a list of tags describing that category.
    Example input:
    { "0": ["laptop", "monitor", "desk"], "1": ["puzzle", "drone", "lego"] }
    
    Procedure to follow:
    1. Go through the categories in order.
    2. Keep a running mental list of every word already used in a previous category name.
    3. Before finalizing each new name, check it against that list.
    4. If a natural word choice would collide, replace it with a more specific or \
    alternative term (synonym, subcategory term, etc.) instead of dropping it silently.
    5. Once all names are generated, double check the full set once more for \
    repeated words before returning the final JSON. If you find a repeat, rename \
    the offending categories.

    Example of collision handling:
    - Category 0 tags: ["desk", "office chair"] -> "Office Furniture"
    - Category 1 tags: ["stapler", "notebook"] -> here "Office" is already used, so \
    instead of "Office Supplies" use "Desk Supplies" or "Stationery"
    
    Rules:
    - Generate one short category name (2-4 words) per key, based on its tags.
    - Return ONLY a JSON object with the old key and the new category name as its value.
    - Keys in the output must match the input keys exactly (as strings).
    - Output language always English.
    Output example:
    { "0": "Electronics & furniture", "1": "Toys & gadgets", "2": 'Supplies' }
"""

system_prompt = { 'role': 'system', 'content': prompt }

def get_categories_cluster_labels(clusters, temperature=0):
    clusters_prompt = { 'role': 'user', 'content': clusters }
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[system_prompt, clusters_prompt],
        temperature=temperature,
    )

    return response.choices[0].message.content

In [4]:
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

In [5]:
from lib.data_load import load_data

df = await load_data()
print(f"\n{'CONCATENATED':<62} {df.shape[0]:>6,} rows x {df.shape[1]} cols")
df.sample(10)

1429_1.csv                                                     34,660 rows
Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products.csv       5,000 rows
Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products_May19.csv 28,332 rows

CONCATENATED                                                   67,992 rows x 28 cols


,id,name,asins,brand,categories,keys,manufacturer,reviews.date,reviews.dateAdded,reviews.dateSeen,...,reviews.userCity,reviews.userProvince,reviews.username,source_file,dateAdded,dateUpdated,primaryCategories,imageURLs,manufacturerNumber,sourceURLs
5796,AVphgVaX1cnluZ0-DR74,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes...",B018Y229OU,Amazon,"Fire Tablets,Tablets,Computers & Tablets,All T...",firetablet7displaywifi8gbincludesspecialoffers...,Amazon,2015-11-24T00:00:00.000Z,2017-05-21T01:10:08Z,"2017-04-30T00:05:00.000Z,2017-06-07T08:10:00.000Z",...,NaN,NaN,Natasha,1429_1.csv,NaN,NaN,NaN,NaN,NaN,NaN
45716,AVpgNzjwLJeJML43Kpxn,AmazonBasics AAA Performance Alkaline Batterie...,"B00QWO9P0O,B00LH3DMUO",Amazonbasics,"AA,AAA,Health,Electronics,Health & Household,C...","amazonbasics/hl002619,amazonbasicsaaaperforman...",AmazonBasics,2015-11-17T00:00:00.000Z,NaN,2017-08-28T00:00:00Z,...,NaN,NaN,ByEdMerk2,Datafiniti_Amazon_Consumer_Reviews_of_Amazon_P...,2015-10-30T08:59:32Z,2019-04-25T09:08:16Z,Health & Beauty,https://images-na.ssl-images-amazon.com/images...,HL-002619,"https://www.barcodable.com/upc/841710106442,ht..."
64922,AVqVGWLKnnc1JgDc3jF1,"Fire Kids Edition Tablet, 7 Display, Wi-Fi, 16...",B018Y23MNM,Amazon,"Fire Tablets,Tablets,All Tablets,Amazon Tablet...",firekidseditiontablet7displaywifi16gbgreenkidp...,Amazon,2017-01-28T00:00:00.000Z,NaN,"2017-06-04T00:00:00Z,2017-06-03T00:00:00Z",...,NaN,NaN,Connie,Datafiniti_Amazon_Consumer_Reviews_of_Amazon_P...,2017-03-03T16:55:53Z,2019-02-25T02:03:34Z,Electronics,https://images-na.ssl-images-amazon.com/images...,53-004683,http://reviews.bestbuy.com/3545/5026300/review...
28742,AVpidLjVilAPnD_xEVpI,NaN,B0189XYY0Q,Amazon Fire,"Electronics,Tablets & E-Readers,Tablets,Back T...","841667101743,amazonfire/51441641,amazonfirehd1...",Amazon,2017-02-15T00:00:00.000Z,NaN,"2017-09-03T08:45:21.110Z,2017-08-27T11:01:00.8...",...,NaN,NaN,ronk1969,1429_1.csv,NaN,NaN,NaN,NaN,NaN,NaN
25613,AVpfl8cLLJeJML43AE3S,"Amazon Fire Tv,,,\r\nAmazon Fire Tv,,,","B00L9EPT8O,B01E6AO69U",Amazon,"Stereos,Remote Controls,Amazon Echo,Audio Dock...","echowhite/263039693056,echowhite/152558276095,...",Amazon,2017-09-20T00:00:00.000Z,NaN,2017-09-28T00:00:00Z,...,NaN,NaN,ray4hall,1429_1.csv,NaN,NaN,NaN,NaN,NaN,NaN
5811,AVphgVaX1cnluZ0-DR74,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes...",B018Y229OU,Amazon,"Fire Tablets,Tablets,Computers & Tablets,All T...",firetablet7displaywifi8gbincludesspecialoffers...,Amazon,2015-11-21T00:00:00.000Z,2017-05-21T01:10:11Z,"2017-04-30T00:05:00.000Z,2017-06-07T08:10:00.000Z",...,NaN,NaN,Shiny151,1429_1.csv,NaN,NaN,NaN,NaN,NaN,NaN
35239,AWFUWc8THh53nbDRF6YO,Amazon Echo Show Alexa-enabled Bluetooth Speak...,B010CEHQTG,Amazon,"Computers,Amazon Echo,Virtual Assistant Speake...","echoshowwhite/ameshowwht,amazon/b010cehqtg,848...",Amazon,2017-10-18T00:00:00.000Z,NaN,2018-04-26T00:00:00Z,...,NaN,NaN,JFBJR,Datafiniti_Amazon_Consumer_Reviews_of_Amazon_P...,2018-02-02T02:30:22Z,2018-10-15T16:03:30Z,"Electronics,Hardware",https://static.bhphoto.com/images/images500x50...,B010CEHQTG,https://reviews.bestbuy.com/3545/5875664/revie...
8353,AVphgVaX1cnluZ0-DR74,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes...",B018Y229OU,Amazon,"Fire Tablets,Tablets,Computers & Tablets,All T...",firetablet7displaywifi8gbincludesspecialoffers...,Amazon,2017-03-17T00:00:00.000Z,2017-05-21T01:27:54Z,"2017-04-30T00:10:00.000Z,2017-06-07T08:15:00.000Z",...,NaN,NaN,cademi,1429_1.csv,NaN,NaN,NaN,NaN,NaN,NaN
24066,AVpfl8cLLJeJML43AE3S,"Echo (White),,,\r\nEcho (White),,,","B00L9EPT8O,B01E6AO69U",Amazon,"Stereos,Remote Controls,Amazon Echo,Audio Dock...","echowhite/263039693056,echowhite/152558276095,...",Amazon,2017-08-25T00:00:00.000Z,NaN,"2017-09-28T00:00:00Z,2017-09-08T00:00:00Z,2017...",...,NaN,NaN,Victor,1429_1.csv,NaN,NaN,NaN,NaN,NaN,NaN
17879,AV1YnRtnglJLPUi8IJmV,Amazon Kindle Paperwhite - eBook reader - 4 GB...,B00OQVZDJM,Amazon,"Walmart for Business,Office Electronics,Tablet...","amazon/b00oqvzdjm,848719056099,amaz

In [6]:
from lib.data_products_clean import clean_data_frame

df = clean_data_frame(df)
print(df.isna().sum())
display(df.sample(10))

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\agcor\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\agcor\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\agcor\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


name                 0
brand                0
asins                0
categories           0
manufacturer         0
reviews.date         0
reviews.username     0
primaryCategories    0
reviews.rating       0
reviews.text         0
reviews.title        0
tags                 0
dtype: int64


,name,brand,asins,categories,manufacturer,reviews.date,reviews.username,primaryCategories,reviews.rating,reviews.text,reviews.title,tags
52717,Kindle Voyage E-reader 6 High-Resolution Displ...,Amazon,B00IOY8XWQ,eBook Readers Electronics Features Walmart Bus...,Amazon,2015-02-04T00:00:00.000Z,ChelseaFan,Electronics,5.0,reader great crisp display excellent battery l...,Great choice,eBook Readers Electronics Features Walmart Bus...
14342,Fire Tablet 7 Display Wi-Fi 8 GB Includes Spec...,Amazon,B018Y229OU,Fire Tablets Tablets Computers Tablets Tablets...,Amazon,2015-11-30T00:00:00.000Z,kato,,4.0,first tablet/iphone confusing learn navigate d...,good tablet price,Fire Tablets Tablets Computers Tablets Tablets...
20550,Kindle Voyage E-reader 6 High-Resolution Displ...,Amazon,B00OQVZDJM,Walmart Business Office Electronics Tablets Of...,Amazon,2017-01-27T00:00:00.000Z,SKPR,,5.0,amazed touch quality feel page turning,Excellent product reading,Walmart Business Office Electronics Tablets Of...
50426,AmazonBasics AA Performance Alkaline Batteries...,Amazon Basics,"B00QWO9P0O,B01IB83NZG,B00MNV8E0C",AA AAA Electronics Features Health Electronics...,AmazonBasics,2016-04-26T00:00:00.000Z,ByAmazon Customer,Health Beauty,5.0,Thanks Amazon great deal batteries seem last l...,Great deal good batteries,AA AAA Electronics Features Health Electronics...
63940,Fire HD 8 Tablet Alexa 8 HD Display 32 GB Tang...,Amazon,B01AHB9C1E,Fire Tablets Tablets Tablets Amazon Tablets Co...,Amazon,2016-12-23T00:00:00.000Z,Jaquil84,Electronics,5.0,think great product would recommend,Great product,Fire Tablets Tablets Tablets Amazon Tablets Co...
41351,AmazonBasics AAA Performance Alkaline Batterie...,Amazon Basics,"B00QWO9P0O,B00LH3DMUO",AA AAA Health Electronics Health Household Cam...,AmazonBasics,2016-10-29T00:00:00.000Z,ByR. Cooper,Health Beauty,5.0,Feel power,Five Stars,AA AAA Health Electronics Health Household Cam...
4780,Fire Tablet 7 Display Wi-Fi 8 GB Includes Spec...,Amazon,B018Y229OU,Fire Tablets Tablets Computers Tablets Tablets...,Amazon,2016-01-02T00:00:00.000Z,Mantis,,4.0,excellent tablet however 's app store good and...,Amazon Fire,Fire Tablets Tablets Computers Tablets Tablets...
2181,All-New Fire HD 8 Tablet 8 HD Display Wi-Fi 16...,Amazon,B01AHB9CN2,Electronics iPad Tablets Tablets Fire Tablets ...,Amazon,2017-01-18T00:00:00.000Z,Nickname,,4.0,Nice tablet great price product makes good gif...,Good gift,Electronics iPad Tablets Tablets Fire Tablets ...
46589,AmazonBasics AAA Performance Alkaline Batterie...,Amazon Basics,"B00QWO9P0O,B00LH3DMUO",AA AAA Health Electronics Health Household Cam...,AmazonBasics,2017-04-25T00:00:00.000Z,Bycyberianmanx,Health Beauty,5.0,honest ... batteries outperformed big name bra...,Quality-Plus Batteries,AA AAA Health Electronics Health Household Cam...
6384,Fire Tablet 7 Display Wi-Fi 8 GB Includes Spec...,Amazon,B018Y229OU,Fire Tablets Tablets Computers Tablets Tablets...,Amazon,2016-04-25T00:00:00.000Z,kathmer,,5.0,Oh love product spend hours playing solitaire ...,WOW,Fire Tablets Tablets Computers Tablets Tablets...


In [8]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.20, random_state=42, stratify=df['brand'])
train_df, val_df = train_test_split(train_df, test_size=0.10, random_state=42, stratify=train_df['brand'])

### Select best K cluster number for K_means training

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from scipy.sparse import hstack

vec_name = TfidfVectorizer(min_df=1)
vec_tags = TfidfVectorizer(min_df=2)

X_train_name = vec_name.fit_transform(train_df['name'])
X_train_tags  = vec_tags.fit_transform(train_df['tags'])

# Set more signaling on product name than his category
X = hstack([X_train_name * 2.0, X_train_tags * 1.0])

k_scores = {}
for k in range(2, min(8, len(train_df["name"]))):
    k_means = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = k_means.fit_predict(X)
    k_score = silhouette_score(X, labels)
    k_scores[k] = k_score
    print(f"k={k} -> inercia={k_means.inertia_:.2f}  silhouette={k_score:.3f}")

best_k = max(k_scores, key=k_scores.get)

k=2 -> inercia=120942.30  silhouette=0.327
k=3 -> inercia=98349.26  silhouette=0.387
k=4 -> inercia=87772.33  silhouette=0.376
k=5 -> inercia=71856.53  silhouette=0.463
k=6 -> inercia=62060.17  silhouette=0.505
k=7 -> inercia=53914.89  silhouette=0.563


In [10]:
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
train_df['cluster'] = kmeans.fit_predict(X)

In [11]:
tags = {}
for cluster in range(best_k):
    tags[cluster] = list(set(sorted(train_df[train_df['cluster'] == cluster]['tags'].sum().split())))

display(tags)

{0: ['E-Readers',
  'Genuine',
  'Brands',
  'Tablets',
  'ElectronicsComputers/Tablets',
  'Kindle',
  'Streaming',
  'Hard',
  'Amazon',
  'E-readers',
  'Tech',
  'ElectronicsFire',
  'Accessories',
  'Electronics',
  'Features',
  'Fire',
  'Drives',
  'Store',
  'Ipads',
  'Categories',
  'Components',
  'Computer',
  'Players',
  'Computers',
  'ElectronicsKindle',
  'ElectronicsTablets',
  'Supplies',
  'ElectronicsComputers',
  'Back',
  'Featured',
  'Devices',
  'College',
  'Music',
  'iPad',
  'Tablet',
  'eBook',
  'ElectronicsElectronics',
  'Office',
  'Frys',
  'Networking',
  'Storage',
  'Toys',
  'Readers',
  'Deals',
  'Media',
  'ElectronicseBook',
  'Movies',
  'Android',
  'Computers/Tablets'],
 1: ['Camcorder',
  'Camera',
  'Accessories',
  'Electronics',
  'Check',
  'Health',
  'Household',
  'Supplies',
  'AA',
  'Batteries',
  'AAA',
  'Care',
  'Robot',
  'Personal',
  'Beauty',
  'Baby',
  'BeautyAA',
  'Photo',
  'Chargers'],
 2: ['Digital',
  'Litter',


In [12]:
import json

openai_clusters_categories_response = get_categories_cluster_labels(json.dumps(tags))
clusters_categories = json.loads(openai_clusters_categories_response)
print(clusters_categories)

{'0': 'Digital Reading Tablets', '1': 'Imaging Power Essentials', '2': 'Home Audio Gear', '3': 'Computing Media Devices', '4': 'Portable Tech Cases', '5': 'Learning Gadget Covers', '6': 'Personal Care Batteries'}


In [13]:
train_df['category'] = train_df['cluster'].apply(lambda cluster: clusters_categories[str(cluster)])
display(train_df.sample(10))

,name,brand,asins,categories,manufacturer,reviews.date,reviews.username,primaryCategories,reviews.rating,reviews.text,reviews.title,tags,cluster,category
3120,Amazon 5W USB Official OEM Charger Power Adapt...,Amazon,B01J2G4VBG,Amazon Devices Accessories Amazon Device Acces...,Amazon,2017-04-06T00:00:00.000Z,Amazon Customer,,5.0,good,Excellent,Amazon Devices Accessories Amazon Device Acces...,2,Home Audio Gear
47066,AmazonBasics AAA Performance Alkaline Batterie...,Amazon Basics,"B00QWO9P0O,B00LH3DMUO",AA AAA Health Electronics Health Household Cam...,AmazonBasics,2016-02-29T00:00:00.000Z,ByBobbi,Health Beauty,5.0,work really well came nice cardboard package o...,work really well came nice cardboard package,AA AAA Health Electronics Health Household Cam...,1,Imaging Power Essentials
15044,Brand New Amazon Kindle Fire 16gb 7 Ips Displa...,Amazon,B018Y225IA,Computers/Tablets Networking Tablets eBook Rea...,Amazon,2016-12-22T00:00:00.000Z,Tisdale0504,,5.0,Allows watch read digitally Great product,Excellent Product Price,Computers/Tablets Networking Tablets eBook Rea...,0,Digital Reading Tablets
43123,AmazonBasics AAA Performance Alkaline Batterie...,Amazon Basics,"B00QWO9P0O,B00LH3DMUO",AA AAA Health Electronics Health Household Cam...,AmazonBasics,2015-08-20T00:00:00.000Z,ByJ. R. G.,Health Beauty,5.0,work 's,Five Stars,AA AAA Health Electronics Health Household Cam...,1,Imaging Power Essentials
19418,Amazon Kindle Paperwhite eBook reader 4 GB 6 m...,Amazon,B00OQVZDJM,Walmart Business Office Electronics Tablets Of...,Amazon,2017-06-03T00:00:00.000Z,rajdesire,,5.0,Best product reading.I using since one half years,Good product,Walmart Business Office Electronics Tablets Of...,3,Computing Media Devices
14687,Brand New Amazon Kindle Fire 16gb 7 Ips Displa...,Amazon,B00IOYAM4I,Computers Tablets E-Readers Accessories eBook ...,Amazon,2015-01-16T00:00:00.000Z,SnowBird61,,5.0,original Kindle filled Upgraded Kindle Voyage ...,Awesome,Computers Tablets E-Readers Accessories eBook ...,0,Digital Reading Tablets
19136,Amazon Kindle Paperwhite eBook reader 4 GB 6 m...,Amazon,B00OQVZDJM,Walmart Business Office Electronics Tablets Of...,Amazon,2017-03-09T00:00:00.000Z,James451,,5.0,fine unit six weeks previously owned three ere...,doncha love nice things,Walmart Business Office Electronics Tablets Of...,3,Computing Media Devices
6070,Fire Tablet 7 Display Wi-Fi 8 GB Includes Spec...,Amazon,B018Y229OU,Fire Tablets Tablets Computers Tablets Tablets...,Amazon,2016-05-27T00:00:00.000Z,Cottagegirl,,5.0,bought keep email computer repaired computer f...,much thought would,Fire Tablets Tablets Computers Tablets Tablets...,4,Portable Tech Cases
35377,Amazon Echo Show Alexa-enabled Bluetooth Speak...,Amazon,B010CEHQTG,Computers Amazon Echo Virtual Assistant Speake...,Amazon,2018-01-25T00:00:00.000Z,Mary,Electronics Hardware,5.0,Bought son wife love thought didn‚Äôt need fin...,Great,Computers Amazon Echo Virtual Assistant Speake...,2,Home Audio Gear
37976,Fire HD 10 Tablet 10.1 HD Display Wi-Fi 16 GB ...,Amazon,B0189XYY0Q,eBook Readers Fire Tablets Electronics Feature...,Amazon,2017-08-18T00:00:00.000Z,Michael1412,Office Supplies Electronics,4.0,Love Amazon however tablet limits using browse...,Decent tablet,eBook Readers Fire Tablets Electronics Feature...,0,Digital Reading Tablets


In [14]:
import os
import joblib

MODEL_DIR = os.path.join(os.getcwd(), 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

model_file = {
    'CLUSTERS_LABELS': clusters_categories,
    'vec_name': vec_name,
    'vec_tags': vec_tags,
    'kmeans': kmeans
}

save_model_path = os.path.join(MODEL_DIR, 'product_category_classifier.joblib')
joblib.dump(model_file, save_model_path)
print(f'Successfully exported to: {save_model_path}')

Successfully exported to: c:\Users\agcor\Develop\ironhack_ai\7_Project3\project-automated-customers-reviews\notebooks\models\product_category_classifier.joblib


In [18]:
import os
import joblib

def predict_category(name, tags):
    ROOT_DIR = os.path.join(os.getcwd())
    ARTIFACT_PATH = os.path.join(ROOT_DIR, 'models', 'product_category_classifier.joblib')
    artifact = joblib.load(ARTIFACT_PATH)

    x_name = artifact['vec_name'].transform([name])
    x_tags = artifact['vec_tags'].transform([tags])
    x_new = hstack([x_name * 2.0, x_tags * 1.0])

    cluster = artifact['kmeans'].predict(x_new)[0]
    return artifact['CLUSTERS_LABELS'][str(cluster)]

predict_category(
    name="Wireless Bluetooth Headphones Noise Cancelling",
    tags="Electronics Audio Headphones"
)

'Home Audio Gear'